 **Opis**:

 Celem modelu jest diagonozowanie wystąpienia cukrzycy na podstawie wskaźników zdrowotnych. Przy pracy nad modelem wykorzystałem dane zbierane w USA znajdujące się na  kaggle pod linkiem : https://www.kaggle.com/datasets/julnazz/diabetes-health-indicators-dataset/data

Wybrałem dataset nazwany ,,diabetes_012'' który zawiera 236,378 odpowiedzi ankietowanych. Zbiór ten jest niezbalansowany i zawiera 21 kolumn z danymi. Kolumna o nazwie ,,diabetes_012'' została wykorzystana przeze mnie jako etykieta [X] w modelu. Zawiera ona 3 klasy:

0 - dla osób bez cukrzycy lub cukrzycy podczas ciąży

1 - dla osób w stanie przedcukrzycowym

2 - dla osób z cukrzycą

  Objaśnienie nazewnicta kolumn z danymi:



*   **HighBP** - Wysokie ciśnienie krwii (High BloodPressure)

  0 - brak wysokiego  ciśnienia krwii

  1 - wysokie  ciśnienie krwii
*   **HighChol** - Wysoki cholesterol

  0 - brak wysokiego cholesterolu

  1 - wysoki cholesterol

*   **CholCheck** - czy osoba mierzyła swój cholesterol w przeciągu ostatnich 5 lat?

  0 - nie

  1 - tak
*   **BMI** - wskaźnik masy ciała (Body Mass Index)
*   **Smoker** - czy osoba wypaliła w swoim życiu przynajmniej 100 papierosów? [ 5 paczek]

  0 - nie
  
  1 - tak
*   **Stroke** - czy osoba kiedykolwiek miała udar?

  0 - nie

  1 - tak
*   **HeartDiseaseorAttack** czy osoba ma chorobe serca lub kiedykolwiek miała zawał?

  0 - nie

  1 - tak
*   **PhysActivity** - czy osoba wykonywała aktywność fizyczna ( nie wliczając w to pracy) w przeciągu ostatnich 30 dni?

  0 - nie

  1 - tak
*   **Fruits** - czy osoba spożywa conajmniej 1 owoc dziennie?

  0 - nie

  1 - tak
*   **Veggies** - czy osoba spożywa conajmniej 1 warzywo dziennie?

  0 - nie

  1 - tak
*   **HvyAlcoholConsump** - osoba spożywająca dużo alkoholu (dorośli mężczyzni 14 drinków tygodniowo i dorosłe kobiety wypijające więcej niż 7 drinków tygodniowo.

  0 - nie

  1 - tak
*   **AnyHealthcare** - osoba posiadajaca dowolny rodzaj ubezpieczenia zdrowotnego.

  0 - nie

  1 - tak
*   **NoDocbcCost** - czy w ciągu ostatnich 12 miesięcy osoba musiała udać się do lekarza, ale nie mogła tego zrobić ze względu na koszty?

  0 - nie

  1 - tak
*   **GenHlth** - odpowiedzi ankietowanych na pytanie "Czy powiedziałbyś, że ogólnie Twoje zdrowie jest następujące: skala 1-5":

 1 - doskonałe

  2 - bardzo dobre
  
  3 - dobre
  
   4 - przeciętne
   
  5 - słabe
*   **MentHlth** -  zdrowie psychiczne osoby w przeciagu ostatnich 30 dni w ilu dniach nie było zbyt dobre
*   **PhysHlth** - zdrowie fizyczne osoby w przeciagu ostatnich 30 dni w ilu dniach nie było zbyt dobre
*   **DiffWalk** - czy osoba ma problem z chodzeniem lub wchodzeniem po schodach?

  0 - nie

  1 - tak
*   **Sex** - płeć

  0 - kobieta
  
  1 - mężczyzna
*   **Age** -  13 poziomowy podział wieku. Skala 1-13:

 1 = 18-24

  8 = 55-59
  
   13 = 80 lub starszy
*   **Education** - poziom wykształcenia. Skala 1-6:

   1 - osoba nigdy nie chodziła do szkoły lub przedszkola
   
   2 - klasy 1 - 8
   
   3 - klasy 9 - 11

   4 - absolwent szkoły średniej

   5 - licencjat

   6 - inżynier lub wyżej


*   **Income** - skala zarobków  1-8 :

   1 - mniej niż $10,000
   
    5 - więcej niż $35,000
    
     11 - $200,000 lub więcej






In [ ]:
# Importy potrzebnych bibliotek

import math
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.under_sampling import NearMiss
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
# Wyznaczenie ścieżki do pliku z danymi
dataset_path = '/content/diabetes.csv'

In [ ]:
# Wczytanie ramki danych
df = pd.read_csv(dataset_path)

In [ ]:
df.head()

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,0,1.0,1,15.0,1.0,0.0,0.0,0,1,...,1,0.0,5.0,10.0,20.0,0.0,0,11,4.0,5.0
1,2.0,1,0.0,1,28.0,0.0,0.0,1.0,0,1,...,1,0.0,2.0,0.0,0.0,0.0,0,11,4.0,3.0
2,2.0,1,1.0,1,33.0,0.0,0.0,0.0,1,1,...,1,0.0,2.0,10.0,0.0,0.0,0,9,4.0,7.0
3,2.0,0,1.0,1,29.0,0.0,1.0,1.0,1,1,...,1,0.0,5.0,0.0,30.0,1.0,1,12,3.0,4.0
4,0.0,0,0.0,1,24.0,1.0,0.0,0.0,0,0,...,1,0.0,3.0,0.0,0.0,1.0,1,13,5.0,6.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236378 entries, 0 to 236377
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Diabetes_012          236378 non-null  float64
 1   HighBP                236378 non-null  int64  
 2   HighChol              236378 non-null  float64
 3   CholCheck             236378 non-null  int64  
 4   BMI                   236378 non-null  float64
 5   Smoker                236378 non-null  float64
 6   Stroke                236378 non-null  float64
 7   HeartDiseaseorAttack  236378 non-null  float64
 8   PhysActivity          236378 non-null  int64  
 9   Fruits                236378 non-null  int64  
 10  Veggies               236378 non-null  int64  
 11  HvyAlcoholConsump     236378 non-null  int64  
 12  AnyHealthcare         236378 non-null  int64  
 13  NoDocbcCost           236378 non-null  float64
 14  GenHlth               236378 non-null  float64
 15  

In [ ]:
df.shape

(236378, 22)

In [ ]:
df.describe()

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
count,236378.000000,236378.000000,236378.000000,236378.000000,236378.000000,236378.000000,236378.000000,236378.000000,236378.000000,236378.000000,...,236378.000000,236378.000000,236378.000000,236378.000000,236378.000000,236378.000000,236378.000000,236378.000000,236378.000000,236378.000000
mean,0.307791,0.418558,0.402059,0.963347,28.953579,0.411997,0.038900,0.086548,0.779231,0.621259,...,0.962573,0.063737,2.480717,3.937710,3.751297,0.153948,0.477824,7.863930,5.139099,6.927451
std,0.705037,0.493324,0.490315,0.187909,6.552055,0.492196,0.193356,0.281172,0.414766,0.485074,...,0.189807,0.244284,1.029134,7.886506,8.245907,0.360900,0.499509,3.236997,0.946185,2.375450
min,0.000000,0.000000,0.000000,0.000000,12.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,1.000000,24.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,1.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,5.000000,4.000000,5.000000
50%,0.000000,0.000000,0.000000,1.000000,28.000000,0.000000,0.000000,0.000000,1.000000,1.000000,...,1.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,8.000000,5.000000,7.000000
75%,0.000000,1.000000,1.000000,1.000000,32.000000,1.000000,0.000000,0.000000,1.000000,1.000000,...,1.000000,0.000000,3.000000,4.000000,2.000000,0.000000,1.000000,10.000000,6.000000,9.000000
max,2.000000,1.000000,1.000000,1.000000,99.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,5.000000,30.000000,30.000000,1.000000,1.000000,13.000000,6.000000,11.000000


In [ ]:
# Sprawdzenie ilości ludzi bez , w stanie przedcukrzycowym oraz z cukrzycą
df['Diabetes_012'].value_counts()

Diabetes_012
0.0    197191
2.0     33568
1.0      5619
Name: count, dtype: int64

In [ ]:
# Sprawdzenie występowanie duplikatów
df.duplicated().sum()

12828

In [ ]:
# Usunięcie duplikatów
df = df.drop_duplicates()

In [ ]:
df.duplicated().sum()

0

In [ ]:
# Sprawdzenie brakujących wartości
print(df.isnull().sum())
# Jako iż dane są kompletne (nie występują żadne wartości Null) możemy przejść dalej

Diabetes_012            0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64


In [ ]:
# Podzielenie zbioru danych na Cechy (X) oraz Etykiety (Y)
X = df.drop('Diabetes_012', axis=1)
y = df['Diabetes_012']

In [ ]:
# Zrównoważenie zbioru danych
nm = NearMiss(version=1, n_neighbors=10)

X,y = nm.fit_resample(X,y)

In [ ]:
X.shape

(16839, 21)

In [ ]:
y.shape

(16839,)

In [ ]:
# Stworzenie obiektu ColumnTransformer, który umożliwia zastosowanie różnych przekształceń do różnych kolumn w DataFrame
preprocessor = ColumnTransformer(
    transformers=[
        # Skalowanie  kolumn niebinarnych za pomocą MinMaxScaler do zakresu [0, 1]
        ('scaler', MinMaxScaler(feature_range=(0, 1)), ['BMI', 'GenHlth', 'MentHlth', 'PhysHlth', 'Age', 'Education', 'Income']),
        # One-hot encoding dla kolumn z danymi kategorycznymi
        ('onehot', OneHotEncoder(),['HighBP','HighChol','CholCheck','Smoker','Stroke','HeartDiseaseorAttack','PhysActivity','Fruits','Veggies','HvyAlcoholConsump','AnyHealthcare','NoDocbcCost','DiffWalk','Sex'])
    ],
    # Pozostawia resztę kolumn bez zmian
    remainder='passthrough'
)

Jako algorytm modelu zdecydowałem się użyć RandomForest, ponieważ ze względu na to że składa się on z wielu drzew decyzyjnych radzi sobie bardzo efektywnie z dużą ilościa danych. Radzi on sobie też z połączeniem danych kategorycznych i liczbowych na których opiera się wykorzystywana przeze mnie ramka danych

In [ ]:
# Stworzenie potoku z transformacja kolumn oraz stworzeniem modelu RandomForest
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(max_depth=16, n_estimators=400, class_weight='balanced'))
])

Do podziału danych wykorzystałem StratifiedShuffleSplit czyli podział warstwowy. Spowodowane jest to niezbalansowanym zbiorem danych przez co jest zdecydowanie mniej danych osób chorych na cukrzyce. Ma to na cel prawidłowe nauczenie modelu diagnozowania cukrzycy.

In [ ]:
splitter = StratifiedShuffleSplit(n_splits=2, test_size=0.2, random_state=42)

In [ ]:
# Podział danych na zbiór treningowy i testowy przy użyciu StratifiedShuffleSplit
for train_index, test_index in splitter.split(X, y):
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]
    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

In [ ]:
# Trenowanie modelu za pomoca danych treningowych przy uzyciu potoku
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('scaler', MinMaxScaler(),
                                                  ['BMI', 'GenHlth', 'MentHlth',
                                                   'PhysHlth', 'Age',
                                                   'Education', 'Income']),
                                                 ('onehot', OneHotEncoder(),
                                                  ['HighBP', 'HighChol',
                                                   'CholCheck', 'Smoker',
                                                   'Stroke',
                                                   'HeartDiseaseorAttack',
                                                   'PhysActivity', 'Fruits',
                                                   'Veggies',
                                                   'HvyAlcoholConsump',
                                                   'AnyHealthcare',
                                                   'NoDocbcCost', 'DiffWalk',
                                                   'Sex'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', max_depth=16,
                                        n_estimators=400))])

In [ ]:
# Przewidzenie wyników dla danych treningowych
X_train_pred = pipeline.predict(X_train)

In [ ]:
# Sprawdzenie dokładności przewidywania dla danych treningowych
accuracy = accuracy_score(X_train_pred, y_train)
print(f'Accuracy: {accuracy}')

Accuracy: 0.8049142602627867


In [ ]:
# Przewidzenie wyników dla danych testowych
X_test_pred = pipeline.predict(X_test)

In [ ]:
# Sprawdzenie dokładności przewidywania dla danych testowych
accuracy = accuracy_score(X_test_pred,y_test)
print(f'Accuracy: {accuracy}')

Accuracy: 0.7366389548693587


In [ ]:
# Stworzenie macierzy pomyłek
conf_matrix = confusion_matrix(X_test_pred,y_test)

In [ ]:
print(conf_matrix)

[[928 121 343]
 [  1 815  42]
 [194 186 738]]


In [ ]:
# Inicjalizacja listy na wartości specyficzności dla każdej klasy
specificities = []
for i in range(conf_matrix.shape[0]):
    # Obliczenie liczby prawdziwych negatywów (TN) dla klasy i
    tn = np.sum(conf_matrix) - (np.sum(conf_matrix[i, :]) + np.sum(conf_matrix[:, i]) - conf_matrix[i, i])
    # Obliczenie liczby fałszywych pozytywów (FP) dla klasy i
    fp = np.sum(conf_matrix[:, i]) - conf_matrix[i, i]
    # Obliczenie specyficzności dla klasy i
    specificity = tn / (tn + fp)
    specificities.append(specificity)

# Średnia specyficzność
average_specificity = np.mean(specificities)

In [ ]:
# Obliczenie precyzji oraz czułości (recall) dla zbioru testowego
precision = precision_score(X_test_pred, y_test, average='weighted')
recall = recall_score(X_test_pred, y_test, average='weighted')

In [ ]:
print(f'Precyzja: {precision}')

Precyzja: 0.744726765581175


In [ ]:
print(f'Czułość: {recall}')

Czułość: 0.7366389548693587


In [ ]:
print(f'Średnia specyficzność: {average_specificity}')

Średnia specyficzność: 0.8692979737968205


In [ ]:
print(f'Specyficzność dla klasy: {specificities}')

Specyficzność dla klasy: [0.9013157894736842, 0.8776892430278884, 0.8288888888888889]


**Analiza wyników z miar:**

**Accuracy** - otrzymaliśmy 0.73 dla danych testowych co oznacza, że 73% obiektów testowych otrzymały poprawną klasyfikację. Różnica między zbiorem treningowym a testowym wynosi 0.07 oznacza to że model nie jest przetrenowany.

**Precision** - otrzymaliśmy 0.74 w zbiorze testowym co oznacza, że 74% pozytywnych przypadków przewidzianych przez model jest rzeczywiscie pozytywnych. Oznacza to, że na wszystkie przypadki przewidzenia stanu przedcukrzycowego lub cukrzycy 74% z nich były prawdziwe.

**Recall** - otrzymaliśmy 0.73 w zbiorze testowym co oznacza, że 73% rzeczywistych pozytywnych przypadków jest poprawnie zidentyfikowana przez model. Natomiast 27% rzeczywistych pozytywnych przypadków jest błędnie klasyfikowana przez model jako negatywne czyli brak cukrzycy.

**Specificity**:

**- dla 0 (brak cukrzycy) -** otrzymaliśmy 0.90 co oznacza, że 90% diagnoz o braku cukrzycy model zidentyfikował poprawnie. Zatem 10% osób w stanie przedcukrzycowym lub z cukrzycą zostało błędnie zidentyfikowane przez model jako stan przedcukrzycowy lub jej wystąpienie.

**- dla 1 (stan przedcukrzycowy) -** otrzymaliśmy 0.87 co oznacza, że 87% diagnoz o stanie przedcukrzycowym model zidentyfikował poprawnie. Zatem 13% osób bez cukrzycy lub z cukrzycą zostało błędnie zidentyfikowane przez model jako brak cukrzycy lub jej wystąpienie.

**- dla 2 (cukrzyca) -** otrzymaliśmy 0.82 co oznacza, że 82% diagnoz o wystąpieniu cukrzycy model zidentyfikował poprawnie. Zatem 18% osób z cukrzycą zostały błędnie zidentyfikowane przez model jako jej całkowity brak lub stan przedcukrzycowy.

In [ ]:
# Słownik definicji dystrybucji hiperparametrów dla RandomForestClassifier
hiperparam_distributions = {
    'classifier__n_estimators': [100, 200, 400, 500],
    'classifier__max_depth': [10, 20, 30, 40, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__max_features': ['sqrt', 'log2'],
    'classifier__bootstrap': [True, False],
    'classifier__criterion': ['gini', 'entropy']
}

In [ ]:
# Konfiguracja RandomizedSearchCV do optymalizacji hiperparametrów potoku
random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=hiperparam_distributions,
    n_iter=25,
    cv=5,
    scoring='accuracy',
    random_state=42
    )

In [ ]:
# Wyszukiwanie najlepszych hiperparametrow
random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(remainder='passthrough',
                                                                transformers=[('scaler',
                                                                               MinMaxScaler(),
                                                                               ['BMI',
                                                                                'GenHlth',
                                                                                'MentHlth',
                                                                                'PhysHlth',
                                                                                'Age',
                                                                                'Education',
                                                                                'Income']),
                                                                              ('onehot',
                                                                               OneHotEncoder(),
                                                                               ['HighBP',
                                                                                'HighChol',
                                                                                'CholCheck',
                                                                                'Smoker',
                                                                                'Stroke',
                                                                                'HeartDiseaseorAttack',
                                                                                'PhysActivity',
                                                                                'Fruits',
                                                                                'Veggies',
                                                                                '...
                   n_iter=25,
                   param_distributions={'classifier__bootstrap': [True, False],
                                        'classifier__criterion': ['gini',
                                                                  'entropy'],
                                        'classifier__max_depth': [10, 20, 30,
                                                                  40, None],
                                        'classifier__max_features': ['sqrt',
                                                                     'log2'],
                                        'classifier__min_samples_leaf': [1, 2,
                                                                         4],
                                        'classifier__min_samples_split': [2, 5,
                                                                          10],
                                        'classifier__n_estimators': [100, 200,
                                                                     400,
                                                                     500]},
                   random_state=42, scoring='accuracy')

In [ ]:
# Najlepsze parametry
random_search.best_params_

{'classifier__n_estimators': 500,
 'classifier__min_samples_split': 10,
 'classifier__min_samples_leaf': 1,
 'classifier__max_features': 'log2',
 'classifier__max_depth': None,
 'classifier__criterion': 'entropy',
 'classifier__bootstrap': True}

In [ ]:
# Przewidywanie etykiety dla zbioru treningowego używając zoptymalizowanego modelu
y_pred = random_search.predict(X_train)

In [ ]:
# Ocena dokładności modelu na zbiorze treningowym poprzez porównanie przewidywanych etykiet z prawdziwymi etykietami
random_search.score(X_train, y_train)

0.8461881077870982

In [ ]:
# Przewidywanie etykiety dla zbioru testowego używając zoptymalizowanego modelu
y_pred = random_search.predict(X_test)

In [ ]:
# Ocena dokładności modelu na zbiorze testowym poprzez porównanie przewidywanych etykiet z prawdziwymi etykietami
random_search.score(X_test, y_test)

0.7422802850356295

In [ ]:
# Stworzenie potoku z wykorzystaniem znalezionych najlepszych parametrow
final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators = 500,criterion = 'gini',max_depth = 20, min_samples_split = 5, min_samples_leaf = 1, max_features = 'log2', bootstrap = True))
])

In [ ]:
# Trenowanie modelu za pomoca danych treningowych przy uzyciu zoptymalizowanego potoku
final_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('scaler', MinMaxScaler(),
                                                  ['BMI', 'GenHlth', 'MentHlth',
                                                   'PhysHlth', 'Age',
                                                   'Education', 'Income']),
                                                 ('onehot', OneHotEncoder(),
                                                  ['HighBP', 'HighChol',
                                                   'CholCheck', 'Smoker',
                                                   'Stroke',
                                                   'HeartDiseaseorAttack',
                                                   'PhysActivity', 'Fruits',
                                                   'Veggies',
                                                   'HvyAlcoholConsump',
                                                   'AnyHealthcare',
                                                   'NoDocbcCost', 'DiffWalk',
                                                   'Sex'])])),
                ('classifier',
                 RandomForestClassifier(max_depth=20, max_features='log2',
                                        min_samples_split=5,
                                        n_estimators=500))])

In [ ]:
# Przewidzenie wyników dla danych treningowych
X_train_pred = final_pipeline.predict(X_train)

In [ ]:
# Sprawdzenie dokładności przewidywania dla danych treningowych
accuracy = accuracy_score(X_train_pred, y_train)
print(f'Accuracy: {accuracy}')

Accuracy: 0.840249424690075


In [ ]:
# Przewidzenie wyników dla danych testowych
X_test_pred = pipeline.predict(X_test)

In [ ]:
# Sprawdzenie dokładności przewidywania dla danych testowych
accuracy = accuracy_score(X_test_pred,y_test)
print(f'Accuracy: {accuracy}')

Accuracy: 0.7366389548693587
